In [8]:
import gc
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [9]:
def fill_zero_rv_with_sampling(df):
    df_filled = df.copy()
    
    for col in df.columns:
        zero_mask = (df[col] == 0)
        num_zeros = zero_mask.sum()
        
        if num_zeros > 0:
            # 0이 아닌 값들만 추출
            non_zero_values = df.loc[~zero_mask, col].values
            # 0이 아닌 값들 중 랜덤하게 복원 추출
            random_samples = np.random.choice(non_zero_values, size=num_zeros)
            df_filled.loc[zero_mask, col] = random_samples
            
    print("각 자산별 기존 분포를 활용해 0값을 채웠습니다.")
    return df_filled


# 2. 다양한 RV 추정량 계산 함수 정의
def calculate_rvs(O, H, L, C):
    rv_dict = {}
    
    # (1) Parkinson (1980) - High-Low 기반
    # 공식: 1 / (4 * ln2) * (ln(H/L))^2
    rv_dict['Parkinson'] = (1 / (4 * np.log(2))) * (np.log(H / L)**2)
    
    # (2) Garman-Klass (1980) - OHLC 기반
    # 공식: 0.5 * (ln(H/L))^2 - (2*ln2 - 1) * (ln(C/O))^2
    rv_dict['Garman_Klass'] = 0.5 * (np.log(H / L)**2) - (2 * np.log(2) - 1) * (np.log(C / O)**2)
    
    # (3) Rogers-Satchell (1991) - Drift-Independent
    # 공식: ln(H/C)*ln(H/O) + ln(L/C)*ln(L/O)
    rv_dict['Rogers_Satchell'] = np.log(H / C) * np.log(H / O) + np.log(L / C) * np.log(L / O)
    
    # (4) Simple Close-to-Close Volatility (비교용)
    # 5일 이동평균 분산 (일반적으로 쓰이는 방식)
    returns = np.log(C / C.shift(1))
    rv_dict['Close_to_Close'] = returns**2
    
    return rv_dict

In [10]:
# # # 1. 대상 티커 리스트 정의 (Yahoo Finance 기준)
# tickers = {
#     "DJI": "^DJI",
#     "NASDAQ": "^IXIC",
#     "HSI": "^HSI",
#     "GOLD": "GC=F",
#     "WTI": "CL=F"
# }

# # 2. 데이터 다운로드 (최근 1년치 일간 데이터)
# # group_by='column'을 사용하면 Open, High, Low, Close 별로 정렬됩니다.
# data = yf.download(list(tickers.values()), start='2000-01-01', end='2025-12-31' , interval="1d", auto_adjust=False)

# # data['Open'].to_csv('data/indices_O.csv')
# # data['High'].to_csv('data/indices_H.csv')
# # data['Low'].to_csv('data/indices_L.csv')
# # data['Close'].to_csv('data/indices_C.csv')
# # data['Volume'].to_csv('data/indices_V.csv')

In [11]:
data_lst = [
    "data/indices_O.csv",
    "data/indices_H.csv",
    "data/indices_L.csv",
    "data/indices_C.csv",
    "data/indices_V.csv",
]

df_JPM = pd.read_csv("data/JPM.csv").set_index('DlyCalDt').sort_index()
df_JPM = df_JPM[['DlyOpen', 'DlyHigh', 'DlyLow', 'DlyClose', 'DlyVol']]
df_JPM.columns = ['O', 'H', 'L', 'C', 'V']

data_dict = {}
for file in data_lst:
    name = file.split("_")[1].split(".")[0]
    data_dict[name] = pd.read_csv(file, index_col=0)
    data_dict[name] = data_dict[name][["^DJI", "^IXIC", "^HSI", "GC=F", "CL=F"]]
    data_dict[name] = pd.concat([data_dict[name], df_JPM[name]], axis=1)
    data_dict[name].columns = ["DJI", "NASDAQ", "HSI", "GOLD", "WTI_OIL", 'JPM']
    data_dict[name] = data_dict[name][["DJI", "NASDAQ", 'JPM', "HSI", "GOLD", "WTI_OIL"]]
    data_dict[name] = data_dict[name].dropna()
    data_dict[name] = data_dict[name].loc["2001-01-02":"2024-12-31", :]
    data_dict[name].index = pd.to_datetime(data_dict[name].index)

In [ ]:
# 1. 기초 데이터 설정 (기존 data_dict 활용)
O = data_dict['O']
H = data_dict['H']
L = data_dict['L']
C = data_dict['C']


# RV 데이터 생성
rv_results = calculate_rvs(O, H, L, C)

c:\Users\asas4\AppData\Local\miniconda3\envs\research\lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


In [20]:
for rv_name in rv_results.keys():
    print(rv_name)
    rv_results[rv_name] = fill_zero_rv_with_sampling(rv_results[rv_name])
    rv_results[rv_name].to_csv(f'RV_{rv_name}.csv')

Parkinson
각 자산별 기존 분포를 활용해 0값을 채웠습니다.
Garman_Klass
각 자산별 기존 분포를 활용해 0값을 채웠습니다.
Rogers_Satchell
각 자산별 기존 분포를 활용해 0값을 채웠습니다.
Close_to_Close
각 자산별 기존 분포를 활용해 0값을 채웠습니다.


In [ ]:
# --- 1. 파라미터 설정 ---
rv_window = 20  # 미래 윈도우 사이즈
epsilon = 1e-12
assets = ["DJI", "NASDAQ", "JPM", "HSI", "GOLD", "WTI_OIL"]
standardized_dict = {}

# ADF 결과를 저장하기 위한 빈 리스트
adf_results_list = []

for rv_name in rv_results.keys():
    print(f"\n{'='*20} Analyzing with RV Method: {rv_name} {'='*20}")
    
    # 6개 자산 통합 시각화 (각 RV 모델별로 Figure 생성)
    fig, axes = plt.subplots(len(assets), 3, figsize=(18, 4 * len(assets)))
    
    for i, asset in enumerate(assets):
        # (1) Future Window RV calculation
        var_path = rv_results[rv_name][asset]
        future_rv = np.sqrt(
            var_path.rolling(window=rv_window, min_periods=1)
            .mean()
            .shift(-(rv_window - 1))
        ).clip(lower=epsilon)
        
        # (2) Log Return & Normalization (Bulk)
        ret = np.log(data_dict['C'][asset] / data_dict['C'][asset].shift(1))
        bulk = (ret / future_rv).replace([np.inf, -np.inf], np.nan).dropna().clip(-10, 10)
        standardized_dict[f"{asset}_{rv_name}"] = bulk
        # (3) ADF Test for Stationary Table
        adf_stat, p_value, _, _, _, _ = adfuller(bulk)
        adf_results_list.append({
            'Asset': asset,
            'RV_Method': rv_name,
            'ADF_Stat': adf_stat,
            'p_value': p_value
        })

        # --- Visualization (Titles in English) ---
        # [Column 1] Time Series
        axes[i, 0].plot(bulk, lw=0.5, color='steelblue')
        axes[i, 0].set_title(f"[{rv_name}] {asset} Std Return", fontsize=12)
        axes[i, 0].axhline(0, color='red', linestyle='--', alpha=0.3)
        
        # [Column 2] ACF
        plot_acf(bulk, lags=40, ax=axes[i, 1], title=f"{asset} ACF ({rv_name})")
        
        # [Column 3] PACF
        plot_pacf(bulk, lags=40, ax=axes[i, 2], title=f"{asset} PACF ({rv_name})")

    plt.tight_layout()
    plt.close()
    gc.collect()

In [ ]:
# --- 2. Stationarity Information Summary Table ---
# 행은 자산(Asset), 열은 RV 방법론(RV_Method)으로 재구조화
df_adf_all = pd.DataFrame(adf_results_list)
df_stationarity_table = df_adf_all.pivot(index='Asset', columns='RV_Method', values='ADF_Stat')

print("\n" + "="*20 + " ADF Test p-values (Stationarity Table) " + "="*20)
print(df_stationarity_table)

# df_stationarity_table.to_csv('adf_result.csv')

In [ ]:
from statsmodels.tsa.api import VAR


# 1. 데이터 병합 (자산별 Series를 하나의 DataFrame으로)
df_all_bulk = pd.DataFrame(standardized_dict).dropna()
assets = list(standardized_dict.keys())

# 2. Train/Test Split (90% 학습, 10% 예측)
train_size = int(len(df_all_bulk) * 0.9)
train, test = df_all_bulk.iloc[:train_size], df_all_bulk.iloc[train_size:]

model_comparison = []

# --- Model A: Random Walk (Baseline) ---
# t시점의 예측값은 t-1 시점의 실제값 (Naive)
rw_forecast = test.shift(1)
# 첫 번째 행은 직전 실제값으로 채움
rw_forecast.iloc[0] = train.iloc[-1]

for asset in assets:
    mse_rw = mean_squared_error(test[asset], rw_forecast[asset])
    model_comparison.append({'Asset': asset, 'Model': 'Random Walk', 'MSE': mse_rw})

# --- Model B: VAR (Multivariate Model) ---
# 최적 Lag 선택 (AIC 기준)
var_model = VAR(train)
order_selection = var_model.select_order(maxlags=15)
optimal_lag = order_selection.aic
print(f"✅ VAR Optimal Lag: {optimal_lag}")

# 모델 학습
var_fitted = var_model.fit(optimal_lag)

# 예측 (Out-of-sample)
# VAR은 이전 lag 정보를 바탕으로 test 기간 전체를 1-step씩 rolling 하거나 한 번에 예측
var_forecast_values = var_fitted.forecast(train.values[-optimal_lag:], steps=len(test))
df_var_forecast = pd.DataFrame(var_forecast_values, index=test.index, columns=assets)

for asset in assets:
    mse_var = mean_squared_error(test[asset], df_var_forecast[asset])
    model_comparison.append({'Asset': asset, 'Model': 'VAR', 'MSE': mse_var})

# --- 3. 결과 요약 및 출력 ---
df_results = pd.DataFrame(model_comparison)
mse_pivot = df_results.pivot(index='Asset', columns='Model', values='MSE')

print("\n" + "="*20 + " Model Comparison: MSE " + "="*20)
print(mse_pivot)

# 성능 개선율 계산 (%)
mse_pivot['Improvement(%)'] = (1 - mse_pivot['VAR'] / mse_pivot['Random Walk']) * 100
print("\n" + "="*20 + " VAR Improvement over RW " + "="*20)
print(mse_pivot[['Improvement(%)']])

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np

# 1. 데이터 병합 (자산별 Series를 하나의 DataFrame으로)
df_all_bulk = pd.DataFrame(standardized_dict).dropna()
assets = list(standardized_dict.keys())

# 2. Train/Test Split (90% 학습, 10% 예측)
train_size = int(len(df_all_bulk) * 0.9)
train, test = df_all_bulk.iloc[:train_size], df_all_bulk.iloc[train_size:]

model_comparison = []

# --- Model A: Random Walk (Baseline) ---
rw_forecast = test.shift(1)
rw_forecast.iloc[0] = train.iloc[-1]

for asset in assets:
    mse_rw = mean_squared_error(test[asset], rw_forecast[asset])
    model_comparison.append({'Asset': asset, 'Model': 'Random Walk', 'MSE': mse_rw})

# --- Model B: VAR (Multivariate Model) ---
var_model = VAR(train)
order_selection = var_model.select_order(maxlags=15)
optimal_lag = order_selection.aic
print(f"✅ VAR Optimal Lag: {optimal_lag}")

var_fitted = var_model.fit(optimal_lag)
var_forecast_values = var_fitted.forecast(train.values[-optimal_lag:], steps=len(test))
df_var_forecast = pd.DataFrame(var_forecast_values, index=test.index, columns=assets)

for asset in assets:
    mse_var = mean_squared_error(test[asset], df_var_forecast[asset])
    model_comparison.append({'Asset': asset, 'Model': 'VAR', 'MSE': mse_var})

# --- 3. 결과 요약 및 출력 ---
df_results = pd.DataFrame(model_comparison)
mse_pivot = df_results.pivot(index='Asset', columns='Model', values='MSE')

print("\n" + "="*20 + " Model Comparison: MSE " + "="*20)
print(mse_pivot)

mse_pivot['Improvement(%)'] = (1 - mse_pivot['VAR'] / mse_pivot['Random Walk']) * 100
print("\n" + "="*20 + " VAR Improvement over RW " + "="*20)
print(mse_pivot[['Improvement(%)']])

# --- 4. 시각화 (Refactored: Graphs) ---
fig, axes = plt.subplots(len(assets), 1, figsize=(14, 3 * len(assets)), sharex=True)

for i, asset in enumerate(assets):
    axes[i].plot(test.index, test[asset], label='Actual Bulk', color='black', alpha=0.3, lw=1)
    axes[i].plot(test.index, rw_forecast[asset], label='Random Walk', color='red', linestyle='--', alpha=0.6)
    axes[i].plot(test.index, df_var_forecast[asset], label=f'VAR (Lag={optimal_lag})', color='blue', lw=1.5)
    
    axes[i].set_title(f"Forecast Comparison: {asset}", fontsize=12)
    axes[i].set_ylabel("Std Return")
    axes[i].legend(loc='upper right', fontsize=9)
    axes[i].grid(True, alpha=0.2)

plt.xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. 폰트 및 스타일 설정 (깔끔한 스타일)
plt.style.use('seaborn-v0_8-whitegrid') # 또는 'ggplot'

# 2. 주요 반응을 볼 자산만 선택 (너무 많으면 다시 뭉개짐)
# 예: DJI 충격에 대해 각 자산의 'Close_to_Close' 정규화 수익률만 확인
# target_responses = [col for col in var_fitted.names if 'Close_to_Close' in col]

# 3. 그래프 출력
# figsize: (가로, 세로)를 충분히 크게 잡아야 글자가 안 겹침
# lty: 라인 스타일, lw: 라인 두께
fig = irf.plot(
    impulse='DJI_Close_to_Close', 

    figsize=(12, 16),      # 세로 길이를 충분히 확보
    subplot_params={'fontsize': 10},
    plot_params={'lw': 2}  # 선 두께 조절
)

# 4. 세부 디자인 조정
plt.subplots_adjust(hspace=0.6, top=0.95) # 그래프 간 간격(hspace) 넓히기
plt.suptitle("Impulse Response: Impact of DJI Shock on All Assets", fontsize=16, fontweight='bold')

# 각 서브플롯의 제목(Title) 정리 (가독성 향상)
for ax in fig.get_axes():
    title = ax.get_title()
    # "DJI_Close_to_Close -> JPM_Close_to_Close" 형태를 "Shock to JPM" 등으로 단순화 가능
    clean_title = title.split('->')[-1].strip()
    ax.set_title(f"Response of {clean_title}", fontsize=11, color='navy')
    ax.axhline(0, color='red', linestyle='--', linewidth=0.8) # 0 기준선 강조

plt.show()